# H121 ISMN Skill Figures

Rerunnable figure package for the ISMN OL/DA soil-moisture skill bundle.

Map and averaging conventions for this decision package:

- station maps are clipped to latitude >= -60 degrees;
- station maps use Robinson projection when Cartopy is available;
- station and network means use the GEOSldas tile area associated with each station's mapped tile;
- paired deltas are signed so positive always means the comparison run is better.

Outputs are written to `projects/ascat_da/output/h121_ismn_skill_figures/` as PNG plus CSV summary tables.



In [ ]:
from __future__ import annotations

from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap
from IPython.display import display

try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    HAS_CARTOPY = True
except Exception as exc:
    cfeature = None
    HAS_CARTOPY = False
    warnings.warn(f"Cartopy unavailable; map cells will fall back to lon/lat axes: {exc}")


mpl.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 300,
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "legend.fontsize": 8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
})

ROOT = Path.cwd()
while ROOT.name != "geosldas-analysis" and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if ROOT.name != "geosldas-analysis":
    raise RuntimeError("Run from inside the geosldas-analysis repository")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from projects.iv_tc.iv_tc.readers import read_tilecoord

BUNDLE = ROOT / "data" / "ismn_ol_da_skill_bundle"
TILECOORD = ROOT / "projects" / "obs_scaling_params" / "test_data" / "inputs" / "OLv7_M36_MULTI_type_13_H121.ldas_tilecoord.bin"
OUT = ROOT / "projects" / "ascat_da" / "output" / "h121_ismn_skill_figures"
OUT.mkdir(parents=True, exist_ok=True)

MAP_LAT_MIN = -60.0
MAP_EXTENT = [-180.0, 180.0, MAP_LAT_MIN, 90.0]
LAND_FACE = "0.92"
ZERO_NEUTRAL_FRACTION = 0.02
RUN_ORDER = ["OL", "DA_legacy", "DA_H121", "DA_SMAP_comb_fp_scaled"]
DA_RUNS = ["DA_legacy", "DA_H121", "DA_SMAP_comb_fp_scaled"]
RUN_LABELS = {
    "OL": "OL",
    "DA_legacy": "DA legacy",
    "DA_H121": "DA H121",
    "DA_SMAP_comb_fp_scaled": "DA SMAP",
}
DOMAIN_ORDER = ["surface", "rz"]
DOMAIN_LABELS = {"surface": "Surface", "rz": "Root zone"}
METRICS = ["R", "anomR", "ubRMSE"]
METRIC_LABELS = {
    "R": "R",
    "anomR": "Anomaly R",
    "ubRMSE": "ubRMSE",
}
METRIC_DELTA_LABELS = {
    "R": "Delta R (run - OL)",
    "anomR": "Delta anomaly R (run - OL)",
    "ubRMSE": "Delta ubRMSE (OL - run)",
}
METRIC_UNITS = {"R": "-", "anomR": "-", "ubRMSE": "m3/m3"}
COLORS = {"OL": "0.55", "DA_legacy": "#0072B2", "DA_H121": "#D55E00", "DA_SMAP_comb_fp_scaled": "#009E73"}

if not BUNDLE.exists():
    raise FileNotFoundError(BUNDLE)
if not TILECOORD.exists():
    raise FileNotFoundError(TILECOORD)




## Shared Helpers



In [ ]:
def save_fig(fig, name: str) -> None:
    png = OUT / f"{name}.png"
    fig.savefig(png, bbox_inches="tight", pad_inches=0.04)
    print(f"saved {png.relative_to(ROOT)}")
    display(fig)

def weighted_stats(values, weights, alpha=0.05) -> dict:
    v = np.asarray(values, dtype=float)
    w = np.asarray(weights, dtype=float)
    ok = np.isfinite(v) & np.isfinite(w) & (w > 0)
    if not ok.any():
        return {"n": 0, "n_eff": np.nan, "mean": np.nan, "ci95_half": np.nan, "sem": np.nan}
    v = v[ok]
    w = w[ok]
    mean = float(np.sum(v * w) / np.sum(w))
    n = int(v.size)
    n_eff = float((np.sum(w) ** 2) / np.sum(w ** 2))
    if n <= 1 or n_eff <= 1:
        return {"n": n, "n_eff": n_eff, "mean": mean, "ci95_half": np.nan, "sem": np.nan}
    var = float(np.sum(w * (v - mean) ** 2) / np.sum(w))
    var *= n_eff / max(n_eff - 1.0, 1.0)
    sem = float(np.sqrt(var / n_eff))
    ci95 = float(stats.t.ppf(1 - alpha / 2, df=max(n_eff - 1.0, 1.0)) * sem)
    return {"n": n, "n_eff": n_eff, "mean": mean, "ci95_half": ci95, "sem": sem}


def paired_delta(metric: str, reference, values):
    reference = np.asarray(reference, dtype=float)
    values = np.asarray(values, dtype=float)
    if metric == "bias":
        return np.abs(reference) - np.abs(values)
    if metric in {"RMSE", "ubRMSE", "MSE", "ubMSE"}:
        return reference - values
    return values - reference


def load_tile_area() -> np.ndarray:
    tc = read_tilecoord(TILECOORD)
    return np.asarray(tc["area"], dtype=float)


tile_area = load_tile_area()
print(f"Loaded tile areas for {tile_area.size:,} GEOSldas tiles")


def add_station_area(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    tile_idx = out["tile_index"].to_numpy(dtype=int)
    area = np.full(len(out), np.nan, dtype=float)
    ok = (tile_idx >= 0) & (tile_idx < tile_area.size)
    area[ok] = tile_area[tile_idx[ok]]
    out["tile_area"] = area
    return out


def setup_map(ax, title: str | None = None):
    if HAS_CARTOPY:
        ax.set_extent(MAP_EXTENT, crs=ccrs.PlateCarree())
        ax.add_feature(cfeature.LAND, facecolor=LAND_FACE, edgecolor="none", zorder=0)
        ax.coastlines(linewidth=0.35, color="0.25", zorder=2)
    else:
        ax.set_facecolor(LAND_FACE)
        ax.set_xlim(MAP_EXTENT[0], MAP_EXTENT[1])
        ax.set_ylim(MAP_EXTENT[2], MAP_EXTENT[3])
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")
    if title:
        ax.set_title(title)


def _panel_label_text(index: int) -> str:
    letters = []
    index = int(index)
    while True:
        letters.append(chr(ord("a") + index % 26))
        index = index // 26 - 1
        if index < 0:
            break
    return f"({''.join(reversed(letters))})"


def add_panel_labels(axes, *, x: float = 0.02, y: float = 0.98) -> None:
    for i, ax in enumerate(np.ravel(np.asarray(axes, dtype=object))):
        if ax is None or not ax.get_visible():
            continue
        ax.text(
            x, y, _panel_label_text(i),
            transform=ax.transAxes,
            ha="left", va="top",
            fontsize=10, fontweight="bold",
            bbox=dict(boxstyle="square,pad=0.15", facecolor="white", edgecolor="none", alpha=0.78),
            zorder=20,
        )


def segmented_cmap_norm(cmap_name: str, vlim: float, n_bins: int = 12, neutral_fraction: float = ZERO_NEUTRAL_FRACTION):
    side_bins = max(int(n_bins) // 2, 1)
    color_bins = side_bins * 2
    neutral = max(float(vlim) * float(neutral_fraction), np.finfo(float).eps)
    neutral = min(neutral, float(vlim) * 0.5)
    neg_bounds = np.linspace(-vlim, -neutral, side_bins + 1)
    pos_bounds = np.linspace(neutral, vlim, side_bins + 1)
    bounds = np.r_[neg_bounds, pos_bounds]
    base = plt.get_cmap(cmap_name, color_bins)
    base_colors = base(np.linspace(0, 1, color_bins))
    colors = np.vstack([
        base_colors[:side_bins],
        np.array([[1.0, 1.0, 1.0, 1.0]]),
        base_colors[side_bins:],
    ])
    cmap = ListedColormap(colors, name=f"{cmap_name}_segmented_zero")
    norm = BoundaryNorm(bounds, cmap.N, clip=True)
    return cmap, norm


def scatter_station_map(ax, df: pd.DataFrame, value_col: str, *, title: str, vlim: float, s=13):
    plot = df[np.isfinite(df[value_col]) & np.isfinite(df["lat"]) & np.isfinite(df["lon"]) & (df["lat"] >= MAP_LAT_MIN)].copy()
    cmap_obj, norm = segmented_cmap_norm("RdBu_r", vlim)
    kwargs = dict(c=plot[value_col], s=s, cmap=cmap_obj, norm=norm, edgecolors="0.15", linewidths=0.15, alpha=0.92)
    if HAS_CARTOPY:
        sc = ax.scatter(plot["lon"], plot["lat"], transform=ccrs.PlateCarree(), **kwargs)
    else:
        sc = ax.scatter(plot["lon"], plot["lat"], **kwargs)
    st = weighted_stats(plot[value_col], plot["tile_area"])
    setup_map(ax, f"{title}\nmean={st['mean']:.3f}; n={st['n']:,}")
    return sc



## Load CSV Bundle and Build Paired Tables



In [ ]:
stations = pd.read_csv(BUNDLE / "ismn_skill_stations.csv")
inventory = pd.read_csv(BUNDLE / "ismn_station_inventory.csv")
network_summary_from_bundle = pd.read_csv(BUNDLE / "ismn_skill_network_summary.csv")
stations = add_station_area(stations)
stations = stations[(stations["lat"] >= MAP_LAT_MIN) & np.isfinite(stations["tile_area"]) & (stations["tile_area"] > 0)].copy()
print(f"Skill rows after map/area filter: {len(stations):,}")
print(f"Unique stations: {stations['station_key'].nunique():,}; networks: {stations['network'].nunique():,}")

# Wide station table, one row per station/domain. Coordinates and tile area come from the OL row where possible.
base_cols = ["station_key", "domain", "network", "station", "lat", "lon"]
base = (
    stations[stations["run"] == "OL"]
    .drop_duplicates(["station_key", "domain"])
    [[*base_cols, "tile_area", "N_pairs", "anomN_pairs"]]
    .rename(columns={"N_pairs": "N_pairs_OL", "anomN_pairs": "anomN_pairs_OL"})
)
if base.empty:
    base = stations.drop_duplicates(["station_key", "domain"])[[*base_cols, "tile_area"]]

for metric in ["R", "anomR", "ubRMSE", "RMSE", "bias"]:
    wide = stations.pivot_table(index=["station_key", "domain"], columns="run", values=metric, aggfunc="first")
    for run in RUN_ORDER:
        if run in wide.columns:
            base[f"{metric}_{run}"] = base.set_index(["station_key", "domain"]).index.map(wide[run]).astype(float)

for metric in METRICS:
    for run in DA_RUNS:
        base[f"delta_{metric}_{run}"] = paired_delta(metric, base[f"{metric}_OL"], base[f"{metric}_{run}"])
    base[f"h121_minus_legacy_{metric}"] = paired_delta(metric, base[f"{metric}_DA_legacy"], base[f"{metric}_DA_H121"])

station_skill = base.reset_index(drop=True)
station_skill.to_csv(OUT / "ismn_station_paired_skill_table.csv", index=False)
display(station_skill.head())



## Area-Weighted Summary Tables



In [ ]:
raw_rows = []
for domain in DOMAIN_ORDER:
    subd = station_skill[station_skill["domain"] == domain]
    for run in RUN_ORDER:
        for metric in METRICS:
            st = weighted_stats(subd[f"{metric}_{run}"], subd["tile_area"])
            raw_rows.append({"domain": domain, "run": run, "metric": metric, **st})
raw_summary = pd.DataFrame(raw_rows)
raw_summary.to_csv(OUT / "ismn_raw_skill_area_weighted_summary.csv", index=False)

delta_rows = []
for domain in DOMAIN_ORDER:
    subd = station_skill[station_skill["domain"] == domain]
    for run in DA_RUNS:
        for metric in METRICS:
            st = weighted_stats(subd[f"delta_{metric}_{run}"], subd["tile_area"])
            delta_rows.append({"domain": domain, "run": run, "metric": metric, **st})
delta_summary = pd.DataFrame(delta_rows)
delta_summary.to_csv(OUT / "ismn_delta_vs_ol_area_weighted_summary.csv", index=False)

network_rows = []
for (network, domain), group in station_skill.groupby(["network", "domain"]):
    for run in DA_RUNS:
        for metric in METRICS:
            st = weighted_stats(group[f"delta_{metric}_{run}"], group["tile_area"])
            if st["n"]:
                network_rows.append({"network": network, "domain": domain, "run": run, "metric": metric, **st})
network_delta = pd.DataFrame(network_rows)
network_delta.to_csv(OUT / "ismn_network_delta_vs_ol_area_weighted_summary.csv", index=False)

display(raw_summary)
display(delta_summary)



## Figure 1: Global Paired Delta Skill

Paired station skill changes relative to OL. Positive bars are better for all metrics; ubRMSE is signed as OL minus run.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12.2, 6.6), sharex=True)
for r, domain in enumerate(DOMAIN_ORDER):
    for c, metric in enumerate(METRICS):
        ax = axes[r, c]
        sub = delta_summary[(delta_summary["domain"] == domain) & (delta_summary["metric"] == metric)].set_index("run").reindex(DA_RUNS)
        x = np.arange(len(DA_RUNS))
        ax.bar(x, sub["mean"], yerr=sub["ci95_half"], capsize=3.0, color=[COLORS[run] for run in DA_RUNS], width=0.66)
        ax.axhline(0, color="0.25", linewidth=0.8)
        ax.grid(axis="y", color="0.88", linewidth=0.7)
        ax.set_title(f"{DOMAIN_LABELS[domain]} | {METRIC_LABELS[metric]}")
        ax.set_ylabel(METRIC_DELTA_LABELS[metric])
        ax.set_xticks(x)
        ax.set_xticklabels([RUN_LABELS[run].replace("DA ", "") for run in DA_RUNS], rotation=15, ha="right")
        for xi, (_, row) in enumerate(sub.iterrows()):
            if np.isfinite(row["mean"]):
                ax.text(xi, ax.get_ylim()[0] + 0.04 * (ax.get_ylim()[1] - ax.get_ylim()[0]), f"n={int(row['n'])}", ha="center", va="bottom", fontsize=7, color="0.25")
fig.suptitle("ISMN paired skill changes relative to OL", y=0.98, fontsize=12)
add_panel_labels(axes)
fig.subplots_adjust(top=0.88, bottom=0.12, hspace=0.34, wspace=0.30)
save_fig(fig, "fig01_ismn_global_paired_delta")
plt.close(fig)



## Figure 2: Global Raw Skill Context

Raw ISMN skill by run. Higher R/anomR is better; lower ubRMSE is better.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12.2, 6.6), sharex=True)
for r, domain in enumerate(DOMAIN_ORDER):
    for c, metric in enumerate(METRICS):
        ax = axes[r, c]
        sub = raw_summary[(raw_summary["domain"] == domain) & (raw_summary["metric"] == metric)].set_index("run").reindex(RUN_ORDER)
        x = np.arange(len(RUN_ORDER))
        ax.bar(x, sub["mean"], yerr=sub["ci95_half"], capsize=3.0, color=[COLORS[run] for run in RUN_ORDER], width=0.68)
        ax.grid(axis="y", color="0.88", linewidth=0.7)
        ax.set_title(f"{DOMAIN_LABELS[domain]} | {METRIC_LABELS[metric]}")
        ax.set_ylabel(f"Mean {metric}")
        ax.set_xticks(x)
        ax.set_xticklabels([RUN_LABELS[run].replace("DA ", "") for run in RUN_ORDER], rotation=18, ha="right")
fig.suptitle("ISMN raw model skill against in-situ observations", y=0.98, fontsize=12)
add_panel_labels(axes)
fig.subplots_adjust(top=0.88, bottom=0.14, hspace=0.34, wspace=0.30)
save_fig(fig, "fig02_ismn_global_raw_skill")
plt.close(fig)



## Figure 3: Network Delta Overlay

Network-level paired changes relative to OL for the largest networks. Points above zero indicate improvement.


In [ ]:
# Keep the overlay legible by using the networks with the most paired surface sites.
network_counts = (
    station_skill[station_skill["domain"] == "surface"].groupby("network")["station_key"].nunique().sort_values(ascending=False)
)
NETWORK_LIMIT = 12
networks = network_counts.head(NETWORK_LIMIT).index.tolist()
markers = ["o", "s", "^", "D", "P", "X", "v", "<", ">", "h", "*", "8"]
network_colors = plt.cm.tab20(np.linspace(0, 1, len(networks)))
style = {net: {"marker": markers[i % len(markers)], "color": network_colors[i]} for i, net in enumerate(networks)}

fig, axes = plt.subplots(2, 3, figsize=(13.5, 7.0), sharex=True)
x = np.arange(len(DA_RUNS))
offsets = np.linspace(-0.28, 0.28, len(networks))
for r, domain in enumerate(DOMAIN_ORDER):
    for c, metric in enumerate(METRICS):
        ax = axes[r, c]
        for ni, network in enumerate(networks):
            sub = (
                network_delta[(network_delta["network"] == network) & (network_delta["domain"] == domain) & (network_delta["metric"] == metric)]
                .set_index("run").reindex(DA_RUNS)
            )
            st = style[network]
            ax.errorbar(x + offsets[ni], sub["mean"], yerr=sub["ci95_half"], fmt=st["marker"], ms=4.2, color=st["color"], ecolor="0.55", elinewidth=0.7, capsize=1.5)
        ax.axhline(0, color="0.25", linewidth=0.8)
        ax.grid(axis="y", color="0.90", linewidth=0.7)
        ax.set_title(f"{DOMAIN_LABELS[domain]} | {METRIC_LABELS[metric]}")
        ax.set_ylabel(METRIC_DELTA_LABELS[metric])
        ax.set_xticks(x)
        ax.set_xticklabels([RUN_LABELS[run].replace("DA ", "") for run in DA_RUNS], rotation=15, ha="right")
legend_handles = [plt.Line2D([0], [0], marker=style[n]["marker"], color="none", markerfacecolor=style[n]["color"], markeredgecolor=style[n]["color"], markersize=5, label=f"{n} (n={network_counts[n]})") for n in networks]
fig.legend(handles=legend_handles, loc="lower center", ncol=4, frameon=False, bbox_to_anchor=(0.5, -0.02))
fig.suptitle("Largest ISMN networks: paired skill changes relative to OL", y=0.98, fontsize=12)
add_panel_labels(axes)
fig.subplots_adjust(top=0.88, bottom=0.23, hspace=0.34, wspace=0.30)
save_fig(fig, "fig03_ismn_network_delta_overlay")
plt.close(fig)



## Figure 4: Station Maps of H121 Advantage Over Legacy

Station maps show H121 advantage over legacy. Red/positive means H121 is better; blue/negative means legacy is better. White marks near-zero differences.


In [ ]:
fig, axes = plt.subplots(
    2, 3, figsize=(13.0, 7.0),
    subplot_kw={"projection": ccrs.Robinson()} if HAS_CARTOPY else {},
)
vlims = {"R": 0.12, "anomR": 0.16, "ubRMSE": 0.010}
last_sc_by_col = {}
for r, domain in enumerate(DOMAIN_ORDER):
    for c, metric in enumerate(METRICS):
        ax = axes[r, c]
        col = f"h121_minus_legacy_{metric}"
        sub = station_skill[station_skill["domain"] == domain]
        sc = scatter_station_map(ax, sub, col, title=f"{DOMAIN_LABELS[domain]} | {METRIC_LABELS[metric]}", vlim=vlims[metric])
        last_sc_by_col[c] = sc
for c, metric in enumerate(METRICS):
    cb = fig.colorbar(
        last_sc_by_col[c],
        ax=axes[:, c].tolist(),
        orientation="horizontal",
        fraction=0.045,
        pad=0.055,
        label=f"H121 advantage over legacy: {METRIC_LABELS[metric]} ({METRIC_UNITS[metric]})",
    )
    ticks = np.linspace(-vlims[metric], vlims[metric], 5)
    cb.set_ticks(ticks)
    if metric == "ubRMSE":
        cb.ax.set_xticklabels([f"{t:.3f}" for t in ticks])
    else:
        cb.ax.set_xticklabels([f"{t:.2f}" for t in ticks])
fig.suptitle("ISMN station-level H121-minus-legacy skill", y=0.98, fontsize=12)
add_panel_labels(axes)
fig.subplots_adjust(top=0.86, bottom=0.15, hspace=0.24, wspace=0.12)
save_fig(fig, "fig04_ismn_h121_minus_legacy_station_maps")
plt.close(fig)



## Figure 5: H121 Versus Legacy Paired Station Scatter

Each point compares legacy and H121 improvement at one station. Points above the diagonal favor H121; points below favor legacy.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12.4, 7.2))
for r, domain in enumerate(DOMAIN_ORDER):
    for c, metric in enumerate(METRICS):
        ax = axes[r, c]
        sub = station_skill[(station_skill["domain"] == domain)].copy()
        x = sub[f"delta_{metric}_DA_legacy"].to_numpy(dtype=float)
        y = sub[f"delta_{metric}_DA_H121"].to_numpy(dtype=float)
        w = sub["tile_area"].to_numpy(dtype=float)
        ok = np.isfinite(x) & np.isfinite(y)
        ax.scatter(x[ok], y[ok], s=10, c="0.25", alpha=0.45, linewidths=0)
        lim = np.nanmax(np.abs(np.r_[x[ok], y[ok]]))
        lim = max(lim, 0.02 if metric != "ubRMSE" else 0.004)
        ax.plot([-lim, lim], [-lim, lim], color="#D55E00", linewidth=1.0)
        ax.axhline(0, color="0.75", linewidth=0.6)
        ax.axvline(0, color="0.75", linewidth=0.6)
        ax.set_xlim(-lim, lim)
        ax.set_ylim(-lim, lim)
        ax.set_aspect("equal", adjustable="box")
        better = weighted_stats(y[ok] - x[ok], w[ok])
        ax.set_title(f"{DOMAIN_LABELS[domain]} | {METRIC_LABELS[metric]}\nH121-legacy={better['mean']:.3f}; n={better['n']}")
        ax.set_xlabel("Legacy improvement vs OL")
        ax.set_ylabel("H121 improvement vs OL")
        ax.grid(color="0.90", linewidth=0.7)
fig.suptitle("Per-station paired improvements: H121 versus legacy", y=0.98, fontsize=12)
add_panel_labels(axes)
fig.subplots_adjust(top=0.88, hspace=0.40, wspace=0.30)
save_fig(fig, "fig05_ismn_h121_vs_legacy_station_scatter")
plt.close(fig)



## Figure 6: Network Ranking of H121 Advantage

Networks are sorted by H121 advantage over legacy. Positive/red bars favor H121; negative/blue bars favor legacy.


In [ ]:
rank_specs = [("rz", "anomR"), ("rz", "ubRMSE"), ("surface", "anomR"), ("surface", "ubRMSE")]
fig, axes = plt.subplots(2, 2, figsize=(12.0, 8.2))
for ax, (domain, metric) in zip(axes.ravel(), rank_specs):
    col = f"h121_minus_legacy_{metric}"
    rows = []
    for network, group in station_skill[station_skill["domain"] == domain].groupby("network"):
        st = weighted_stats(group[col], group["tile_area"])
        if st["n"] >= 3:
            rows.append({"network": network, **st})
    rank = pd.DataFrame(rows).sort_values("mean")
    top = pd.concat([rank.head(8), rank.tail(8)]).drop_duplicates("network")
    y = np.arange(len(top))
    ax.barh(y, top["mean"], xerr=top["ci95_half"], color=np.where(top["mean"] >= 0, "#D55E00", "#0072B2"), alpha=0.85)
    ax.axvline(0, color="0.25", linewidth=0.8)
    ax.set_yticks(y)
    ax.set_yticklabels(top["network"])
    ax.set_title(f"{DOMAIN_LABELS[domain]} | {METRIC_LABELS[metric]}")
    ax.set_xlabel("H121 advantage over legacy")
    ax.grid(axis="x", color="0.90", linewidth=0.7)
fig.suptitle("ISMN networks with largest H121/legacy separation", y=0.98, fontsize=12)
add_panel_labels(axes)
fig.subplots_adjust(top=0.88, left=0.16, hspace=0.34, wspace=0.35)
save_fig(fig, "fig06_ismn_network_ranking")
plt.close(fig)



## Supplemental Coverage Figures

Coverage/context figures for the station inventory and pair counts. Colors identify candidate or retained stations, not skill direction.


In [ ]:
# Station inventory and pair-count context.
fig, axes = plt.subplots(
    1, 2, figsize=(12.5, 3.8),
    subplot_kw={"projection": ccrs.Robinson()} if HAS_CARTOPY else {},
)
inv = inventory[np.isfinite(inventory["lat"]) & np.isfinite(inventory["lon"]) & (inventory["lat"] >= MAP_LAT_MIN)]
if HAS_CARTOPY:
    axes[0].scatter(inv["lon"], inv["lat"], transform=ccrs.PlateCarree(), s=8, c="0.65", linewidths=0, alpha=0.70)
    axes[1].scatter(station_skill.drop_duplicates("station_key")["lon"], station_skill.drop_duplicates("station_key")["lat"], transform=ccrs.PlateCarree(), s=8, c="#D55E00", linewidths=0, alpha=0.75)
else:
    axes[0].scatter(inv["lon"], inv["lat"], s=8, c="0.65", linewidths=0, alpha=0.70)
    axes[1].scatter(station_skill.drop_duplicates("station_key")["lon"], station_skill.drop_duplicates("station_key")["lat"], s=8, c="#D55E00", linewidths=0, alpha=0.75)
setup_map(axes[0], f"Candidate inventory\nn={inv['station_key'].nunique():,}")
setup_map(axes[1], f"Skill-filtered stations\nn={station_skill['station_key'].nunique():,}")
fig.suptitle("ISMN station coverage before and after skill filtering", y=0.98, fontsize=12)
add_panel_labels(axes)
save_fig(fig, "figS01_ismn_station_inventory_maps")
plt.close(fig)

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.0), sharey=True)
for ax, domain in zip(axes, DOMAIN_ORDER):
    sub = station_skill[station_skill["domain"] == domain]
    ax.hist(sub["N_pairs_OL"], bins=28, color="#0072B2", alpha=0.75)
    ax.set_title(DOMAIN_LABELS[domain])
    ax.set_xlabel("OL paired days")
    ax.grid(axis="y", color="0.90", linewidth=0.7)
axes[0].set_ylabel("Stations")
fig.suptitle("ISMN paired-sample count distribution", y=0.98, fontsize=12)
add_panel_labels(axes)
save_fig(fig, "figS02_ismn_pair_counts")
plt.close(fig)



## Notes



Error bars in the bar and overlay figures are approximate 95 percent confidence intervals around weighted paired station means. The mean uses mapped GEOSldas tile area. The interval uses the weighted station-delta variance and Kish effective sample size. Deltas are paired by station/domain so each station acts as its own OL control.

